Config and read data

In [98]:
import pandas as pd
import re 

df = pd.read_csv("../data/spam.csv", encoding="latin-1")

df = df[["v1", "v2"]].copy()

df = df.rename(columns={
    "v1": "Category",
    "v2": "Message"
})

df.head()

text_column_name = df['Message']
label_name= df['Category']

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
test_size = 0.2 # data set aside for testing
num_labels = 2


Clean Dataset

In [99]:
def cleaner(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)   # HTML sil
    text = re.sub(r"\s+", " ", text)     # fazla boşluk sil
    return text.strip()


#The apply() method allows you to apply a function along one of the axis of the DataFrame, default 0, which is the index (row) axis.
df["clean_message"] = df['Message'].apply(cleaner)
print(df)
#reading and prapering the data set


     Category                                            Message  \
0         ham  Go until jurong point, crazy.. Available only ...   
1         ham                      Ok lar... Joking wif u oni...   
2        spam  Free entry in 2 a wkly comp to win FA Cup fina...   
3         ham  U dun say so early hor... U c already then say...   
4         ham  Nah I don't think he goes to usf, he lives aro...   
...       ...                                                ...   
5567     spam  This is the 2nd time we have tried 2 contact u...   
5568      ham              Will Ì_ b going to esplanade fr home?   
5569      ham  Pity, * was in mood for that. So...any other s...   
5570      ham  The guy did some bitching but I acted like i'd...   
5571      ham                         Rofl. Its true to its name   

                                          clean_message  
0     go until jurong point, crazy.. available only ...  
1                         ok lar... joking wif u oni...  
2     fre

Label Encoder

In [100]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
le.fit(df['Category'].tolist()) 

df['label'] = le.transform(df['Category'].tolist())

df.head() # label added

,Category,Message,clean_message,label
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",0
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,1
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro...",0


Train/Test Split

In [102]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size= test_size, random_state=42)

Convert to Huggingface Dataset

In [103]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

print(train_dataset)

#df["clean_message"].iloc[0]

Dataset({
    features: ['Category', 'Message', 'clean_message', 'label', '__index_level_0__'],
    num_rows: 4457
})


Tokenizer

In [105]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(example):
    return tokenizer(example["clean_message"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched= True)
print(tokenized_train)


Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Dataset({
    features: ['Category', 'Message', 'clean_message', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4457
})


In [106]:
tokenized_test = test_dataset.map(preprocess_function, batched =True)

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

Init the model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = num_labels)



Train model

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()



In [ ]:
trainer.save_model('spam_model')


Evaluate with classification report

In [ ]:
from sklearn.metrics import classification_report

preds = trainer.predict(tokenized_train)
preds = np.argmax(preds[:3][0], axis =1)

GT= df_train['label'].tolist()

print(classification_report(GT,preds))